In [1]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

powerPlantsDf = pd.read_csv("powerplants.csv")
germanyPowerPlantsDf = powerPlantsDf[powerPlantsDf["Country"] == "Germany"]
germanyPowerPlantsDf.head

<bound method NDFrame.head of             id                             Name   Fueltype      Technology  \
0            0  Pumpspeicherkraftwerk Erzhausen      Hydro  Pumped Storage   
10          10               Psw Goldisthal Pss      Hydro  Pumped Storage   
28          28                  Koepchenwerk Bl      Hydro  Pumped Storage   
29          29                          Waldeck      Hydro  Pumped Storage   
31          31  Pumpspeicherwerk Ronkhausen Psw      Hydro  Pumped Storage   
...        ...                              ...        ...             ...   
164975  165490                        Bielefeld  Hard Coal             NaN   
164981  165496                         Mannheim  Hard Coal             NaN   
164984  165499           Scholven Buer Scholven  Hard Coal             NaN   
164987  165502          Voelklingen Fenne Model  Hard Coal             NaN   
164988  165503                  Wolfsburg North  Hard Coal             NaN   

            Set  Country  Capacit

In [13]:
powerPlantsDf.Fueltype.unique()

<StringArray>
[             'Hydro',          'Hard Coal',        'Natural Gas',
            'Lignite',                'Oil',               'Wind',
      'Solid Biomass',              'Waste',              'Solar',
         'Geothermal',            'Battery',       'Heat Storage',
            'Nuclear',              'Other',             'Biogas',
 'Mechanical Storage',   'Hydrogen Storage']
Length: 17, dtype: str

In [3]:
germanyPowerPlantsDf.Name.to_csv("germany_power_plants_names.csv", index=False)

In [14]:
redispatch_data = pd.read_csv("Redispatch_Daten.csv", sep=";")


In [16]:
redispatch_data.PRIMAERENERGIEART.unique()

<StringArray>
['Sonstiges', 'Konventionell', 'Erneuerbar']
Length: 3, dtype: str

In [9]:
merged = pd.read_csv("powerplants_pypsa_germany_merged.csv")
print(merged.shape[0])
print(merged[merged["Capacity"] > 1].shape[0])

104227
20831


In [3]:
onlyNames = pd.read_csv("germany_power_plants_names.csv")

In [9]:
pd.Series(onlyNames.Name.unique()).to_csv("germany_power_plants_names_unique.csv", index=False)

In [ ]:
redispatch_data = pd.read_csv("Redispatch_Daten.csv", sep=";")
top100 = redispatch_data.head(100)
top100.to_csv("redispatch_top100.csv", index=False, sep=";")

In [3]:
import re


In [5]:
PLANT_PREFIX_RE = re.compile(
    r"^(?:"
    r"Ready|Bhkw|Gtkw|Gud|Hkw|Psw|Kng|Wp|Windpark|Windkraftanlage|Wka|Pv|Kw"
    r")\s+",
    re.IGNORECASE,
)
# ── patterns that identify non-individual-plant entries ───────────────────────
# These cannot be matched to a single row in powerplants.csv.
AGGREGATED_RE = re.compile(
    r"_CR_"                   # control reserve node: 50H_MNS_CR_KLM, AMP_..._CR_WIND
    r"|VNB\b"                 # DSO aggregation bucket
    r"|\bCluster\b"           # wind/PV cluster: AVA Cluster, SHN Cluster
    r"|\s+EE$"                # regional renewable bucket: "EE Bayern", "Baden-Württemberg Nord EE"
    r"|^EE\s"                 # starts with EE: "EE Hessen"
    r"|^Börse$"               # electricity market entry (not a plant)
    r"|^Notfall-RD"           # emergency virtual entries
    r"|^BETROFFENE_ANLAGE$",  # stray CSV header row
    re.IGNORECASE,
)

# ── TSO prefix to strip before fuzzy matching (redispatch query names) ────────
TSO_PREFIX_RE = re.compile(r"^50H\s+", re.IGNORECASE)

# ── parenthesised suffix noise to strip ──────────────────────────────────────
SUFFIX_NOISE_RE = re.compile(r"\s*\((?:KapRes|bnBm|SysRel|kurativ|[A-Za-z\s]+)\)\s*$")

# ── helpers ───────────────────────────────────────────────────────────────────
def normalise_for_matching(name: str) -> str:
    name = TSO_PREFIX_RE.sub("", name)
    name = SUFFIX_NOISE_RE.sub("", name)
    return name.strip()


def normalise_plant_name(name: str) -> str:
    """Strip leading technical type prefixes from powerplant candidate names."""
    prev = None
    while prev != name:
        prev = name
        name = PLANT_PREFIX_RE.sub("", name)
    return name.strip()


In [6]:
germanyPowerPlantsDf.head

<bound method NDFrame.head of             id                             Name   Fueltype      Technology  \
0            0  Pumpspeicherkraftwerk Erzhausen      Hydro  Pumped Storage   
10          10               Psw Goldisthal Pss      Hydro  Pumped Storage   
28          28                  Koepchenwerk Bl      Hydro  Pumped Storage   
29          29                          Waldeck      Hydro  Pumped Storage   
31          31  Pumpspeicherwerk Ronkhausen Psw      Hydro  Pumped Storage   
...        ...                              ...        ...             ...   
164975  165490                        Bielefeld  Hard Coal             NaN   
164981  165496                         Mannheim  Hard Coal             NaN   
164984  165499           Scholven Buer Scholven  Hard Coal             NaN   
164987  165502          Voelklingen Fenne Model  Hard Coal             NaN   
164988  165503                  Wolfsburg North  Hard Coal             NaN   

            Set  Country  Capacit

In [9]:
germanyPowerPlantsDf.DateIn

0         1964.0
10        1970.0
28        1970.0
29        1931.0
31        1969.0
           ...  
164975    1980.0
164981    1966.0
164984    1968.0
164987    1982.0
164988    2000.0
Name: DateIn, Length: 141420, dtype: float64

In [11]:
with open("powerplants_pypsa_germany_merged.csv") as f:                                                                                                                                                                                                                      
    import pandas as pd                                                                                                                                                                                                                                                      
    df = pd.read_csv("powerplants_pypsa_germany_merged.csv")                                                                                                                                                                                                                 
    df[df["Name"].str.contains("Wilhelmshaven", case=False, na=False)][["id","Name","Fueltype","Capacity","DateIn","DateOut"]]             
    print(df[df["Name"].str.contains("Wilhelmshaven", case=False, na=False)][["id","Name","Fueltype","Capacity","DateIn","DateOut"]])

            id                                Name     Fueltype    Capacity  \
87         742                       Wilhelmshaven    Hard Coal  1608.65000   
2044      3204                    Wilhelmshaven Gt          Oil    56.00000   
16844    25002                    Wilhelmshaven On    Hard Coal   788.10000   
25220    44602         Bhkw Klinikum Wilhelmshaven  Natural Gas     0.80000   
27351    47105              Bhkw Zka Wilhelmshaven  Natural Gas     0.50000   
35332    58018          De Wilhelmshaven Potenburg        Solar     0.22000   
60455    94247     Oldebruggestrasse Wilhelmshaven        Solar     0.41976   
61451    95795  Pasm Nea Wilhelmshaven Schillerstr          Oil     0.28800   
69306   107753             Pv Anlage Wilhelmshaven        Solar     0.20100   
69307   107754          Pv Anlage Wilhelmshaven Al        Solar     0.20500   
69308   107755          Pv Anlage Wilhelmshaven Rl        Solar     0.20300   
95693   141463                   Wea Wilhelmshaven  

In [13]:
from plant_match_pre_filtering import load_plants
from datetime import date                                                                                                                                                                                          

plants = load_plants("powerplants_pypsa_germany_merged.csv")                                                                                                                                                       
                                            
wilhelmshaven = [p for p in plants if "wilhelmshaven" in p.name.lower()]                                                                                                                                           
for p in wilhelmshaven:                       
    print(p.id, p.name, p.date_in, p.date_out, p.is_operational(date(2025, 3, 4)))                                                                                                                                 
                                                                                                                                                                                                                    


  [load_plants] 104227 total rows read from powerplants_pypsa_germany_merged.csv
  [load_plants] 104227 rows after Germany filter
742 Wilhelmshaven 1976-01-01 2021-12-31 False
3204 Wilhelmshaven Gt 1973-01-01 2023-12-31 False
25002 Wilhelmshaven On 1976-01-01 2021-12-31 False
44602 Bhkw Klinikum Wilhelmshaven 2016-01-01 None True
47105 Bhkw Zka Wilhelmshaven 2021-01-01 None True
58018 De Wilhelmshaven Potenburg 2020-01-01 None True
94247 Oldebruggestrasse Wilhelmshaven 2018-01-01 None True
95795 Pasm Nea Wilhelmshaven Schillerstr 1988-01-01 None True
107753 Pv Anlage Wilhelmshaven 2010-01-01 None True
107754 Pv Anlage Wilhelmshaven Al 2010-01-01 None True
107755 Pv Anlage Wilhelmshaven Rl 2010-01-01 None True
141463 Wea Wilhelmshaven 2015-01-01 None True
163014 Wp Wilhelmshaven 2000-01-01 None True
164115 Zen Wilhelmshaven 2011-01-01 None True


In [15]:
for p in wilhelmshaven:                       
    print(p.id, p.name, p.date_in, p.date_out, p.capacity_mw, p.fueltype,p.is_operational(date(2025, 3, 4)))                                                                                                                                 
                                                                                                                                                                                                                    


742 Wilhelmshaven 1976-01-01 2021-12-31 1608.65 Hard Coal False
3204 Wilhelmshaven Gt 1973-01-01 2023-12-31 56.0 Oil False
25002 Wilhelmshaven On 1976-01-01 2021-12-31 788.1 Hard Coal False
44602 Bhkw Klinikum Wilhelmshaven 2016-01-01 None 0.8 Natural Gas True
47105 Bhkw Zka Wilhelmshaven 2021-01-01 None 0.5 Natural Gas True
58018 De Wilhelmshaven Potenburg 2020-01-01 None 0.22 Solar True
94247 Oldebruggestrasse Wilhelmshaven 2018-01-01 None 0.4197599999999999 Solar True
95795 Pasm Nea Wilhelmshaven Schillerstr 1988-01-01 None 0.288 Oil True
107753 Pv Anlage Wilhelmshaven 2010-01-01 None 0.201 Solar True
107754 Pv Anlage Wilhelmshaven Al 2010-01-01 None 0.205 Solar True
107755 Pv Anlage Wilhelmshaven Rl 2010-01-01 None 0.203 Solar True
141463 Wea Wilhelmshaven 2015-01-01 None 7.0 Wind True
163014 Wp Wilhelmshaven 2000-01-01 None 8.0 Wind True
164115 Zen Wilhelmshaven 2011-01-01 None 0.238 Solar True


In [1]:
from plant_match_pre_filtering import load_plants
from datetime import date                                                                                                                                                                                          

plants = load_plants("powerplants_pypsa_germany_merged.csv")                                                                                                                                                       
                                            
wilhelmshaven = [p for p in plants if "wilhelmshaven" in p.name.lower()]                                                                                                                                           
for p in wilhelmshaven:                       
    print(p.id, p.name, p.date_in, p.date_out, p.is_operational(date(2025, 3, 4)))                                                                                                                                 
                                                                                                                                                                                                                    

  [load_plants] 104227 total rows read from powerplants_pypsa_germany_merged.csv
  [load_plants] 104227 rows after Germany filter
742 Wilhelmshaven 1976-01-01 2021-12-31 True
3204 Wilhelmshaven Gt 1973-01-01 2023-12-31 True
25002 Wilhelmshaven On 1976-01-01 2021-12-31 True
44602 Bhkw Klinikum Wilhelmshaven 2016-01-01 None True
47105 Bhkw Zka Wilhelmshaven 2021-01-01 None True
58018 De Wilhelmshaven Potenburg 2020-01-01 None True
94247 Oldebruggestrasse Wilhelmshaven 2018-01-01 None True
95795 Pasm Nea Wilhelmshaven Schillerstr 1988-01-01 None True
107753 Pv Anlage Wilhelmshaven 2010-01-01 None True
107754 Pv Anlage Wilhelmshaven Al 2010-01-01 None True
107755 Pv Anlage Wilhelmshaven Rl 2010-01-01 None True
141463 Wea Wilhelmshaven 2015-01-01 None True
163014 Wp Wilhelmshaven 2000-01-01 None True
164115 Zen Wilhelmshaven 2011-01-01 None True


## MaStR Cluster Search

Search MaStR `EinheitenWind.xml` for the top aggregated cluster entries from the redispatch data.
Each cluster name is mapped to a location keyword (Gemeinde / Gemarkung / Ort / Landkreis).

In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd

# cluster name → location keywords to match against Gemeinde/Ort/Gemarkung/Landkreis
CLUSTER_SEARCH = {
    "SHN Cluster Heide West":          ["heide"],
    "SHN Cluster Süderdonn":           ["süderdonn", "suderdonn", "dithmarschen"],
    "SHN Cluster Handewitt":           ["handewitt"],
    "SHN Cluster Husum Nord":          ["husum"],
    "SHN Cluster Klixbüll":            ["klixbüll", "klixbuell", "klixbull"],
    "SHN Cluster Kiel Süd":            ["kiel"],
    "SHN Cluster Krümmel":             ["krümmel", "krummel", "geesthacht"],
    "AVA Cluster Voslapp":             ["voslapp", "wilhelmshaven"],
    "AVA Cluster Helmstedt":           ["helmstedt"],
    "AVA Cluster Lehrte":              ["lehrte"],
    "BAG NWAK-Cluster 17 Altheim":     ["altheim"],
    "BAG NWAK-Cluster 15 Pleinting":   ["pleinting"],
    "BAG NWAK-Cluster 16 Regensburg":  ["sittling", "regensburg"],
}

MASTR_WIND_FILE = "Gesamtdatenexport_20260328_25.2/EinheitenWind.xml"
FIELDS = [
    "EinheitMastrNummer", "NameStromerzeugungseinheit",
    "Gemeinde", "Gemarkung", "Ort", "Landkreis", "Postleitzahl",
    "Bruttoleistung", "Laengengrad", "Breitengrad",
    "Inbetriebnahmedatum", "EinheitBetriebsstatus",
]

def search_mastr_wind(xml_path, cluster_map):
    """Stream-parse EinheitenWind.xml and collect hits per cluster."""
    hits = {cluster: [] for cluster in cluster_map}

    with open(xml_path, "rb") as f:
        for event, elem in ET.iterparse(f, events=("end",)):
            if elem.tag != "EinheitWind":
                continue

            row = {field: (elem.findtext(field) or "").strip() for field in FIELDS}
            search_text = " ".join([
                row["Gemeinde"], row["Gemarkung"], row["Ort"], row["Landkreis"]
            ]).lower()

            for cluster, keywords in cluster_map.items():
                if any(kw in search_text for kw in keywords):
                    hits[cluster].append(row)

            elem.clear()

    return {cluster: pd.DataFrame(rows) for cluster, rows in hits.items()}

print("Parsing MaStR wind file — this takes ~30s …")
results = search_mastr_wind(MASTR_WIND_FILE, CLUSTER_SEARCH)
print("Done.")
for cluster, df in results.items():
    print(f"  {cluster}: {len(df)} turbines")

In [ ]:
# Display results for each cluster — summary + individual turbines
for cluster, df in results.items():
    print(f"\n{'='*70}")
    print(f"  {cluster}  ({len(df)} turbines in MaStR)")
    print(f"{'='*70}")
    if df.empty:
        print("  *** NOT FOUND — PyPSA coverage gap or different location name ***")
        continue
    df["Bruttoleistung"] = pd.to_numeric(df["Bruttoleistung"], errors="coerce")
    print(f"  Total capacity : {df['Bruttoleistung'].sum():.1f} MW")
    print(f"  Gemeinde(n)    : {df['Gemeinde'].unique().tolist()}")
    print(f"  Gemarkung(en)  : {df['Gemarkung'].unique().tolist()[:5]}")
    print()
    print(df[["EinheitMastrNummer", "NameStromerzeugungseinheit", "Gemeinde",
              "Bruttoleistung", "Laengengrad", "Breitengrad",
              "Inbetriebnahmedatum", "EinheitBetriebsstatus"]].to_string(index=False))